In [ ]:
import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

import src.imports as lib

In [ ]:
# ==========================================
# 02 - DATA VALIDATION
# ĐỌC DỮ LIỆU ĐÃ GHÉP
# ==========================================

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import src.imports as lib

file_path = lib.Path("../data/data-clean/hoc_sinh_lop6_9_clean.xlsx")

df_merged = lib.pd.read_excel(
    file_path,
    engine="openpyxl"
)

print(f"Kích thước dữ liệu: {df_merged.shape}")
display(df_merged.head())

In [ ]:
# ==========================================
# 04 - DATA SPLITTING
# THIẾT LẬP RANDOM SEED CỐ ĐỊNH
# ==========================================

import sys
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))
import src.imports as lib

from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
lib.np.random.seed(RANDOM_SEED)

# Đọc dữ liệu đã tích hợp
data_path = lib.Path("../data/data-clean/hoc_sinh_lop6_9_clean.xlsx")
df = lib.pd.read_excel(data_path, engine="openpyxl")

id_col = "ma_hoc_sinh"

choice_cols = [
    "Vật lí", "Hóa học", "Sinh học", "Địa lí",
    "Giáo dục kinh tế và pháp luật (GDKT&PL)",
    "Tin học", "Công nghệ", "Âm nhạc", "Mĩ thuật"
]

score_cols = [
    col for col in df.columns
    if col.startswith(("L6_", "L7_", "L8_", "L9_"))
    and col.endswith(("_hki", "_hkii", "_ca_nam"))
]

assert df.shape == (10000, 106)
assert len(score_cols) == 96
assert df[id_col].is_unique
assert df[id_col].notna().all()

print(f"Random Seed: {RANDOM_SEED}")
print(f"Kích thước dữ liệu: {df.shape}")

In [ ]:
# ==========================================
# CHIA DEVELOPMENT / INDEPENDENT TEST
# ==========================================

dev_ids, test_ids = train_test_split(
    df[id_col],
    test_size=0.10,
    random_state=RANDOM_SEED,
    shuffle=True
)

dev_ids = dev_ids.reset_index(drop=True)
test_ids = test_ids.reset_index(drop=True)

df_dev = df[df[id_col].isin(dev_ids)].copy()
df_test = df[df[id_col].isin(test_ids)].copy()

assert len(df_dev) == 9000
assert len(df_test) == 1000

print("=" * 60)
print("KẾT QUẢ CHIA DỮ LIỆU")
print("=" * 60)

print(f"Development: {len(df_dev):,} học sinh ({len(df_dev)/len(df)*100:.1f}%)")
print(f"Independent Test: {len(df_test):,} học sinh ({len(df_test)/len(df)*100:.1f}%)")

In [ ]:
# ==========================================
# ĐÓNG BĂNG DANH SÁCH INDEPENDENT TEST ID
# ==========================================

split_dir = lib.Path("../data/data-splits")
split_dir.mkdir(parents=True, exist_ok=True)

test_id_file = split_dir / "independent_test_ids.csv"

if test_id_file.exists():

    saved_ids = lib.pd.read_csv(
        test_id_file,
        encoding="utf-8-sig"
    )[id_col]

    assert set(saved_ids) == set(test_ids), (
        "Independent Test ID đã lưu khác với lần chia hiện tại!"
    )

    print("PASS: Danh sách Independent Test ID không thay đổi.")

else:

    test_ids.to_frame(name=id_col).to_csv(
        test_id_file,
        index=False,
        encoding="utf-8-sig"
    )

    print("PASS: Đã lưu cố định Independent Test ID.")

print(f"Đường dẫn: {test_id_file.resolve()}")
print(f"Số lượng ID: {len(test_ids):,}")

In [ ]:
# ==========================================
# KIỂM TRA DATA LEAKAGE THEO ID
# ==========================================

dev_id_set = set(df_dev[id_col])
test_id_set = set(df_test[id_col])

overlap_ids = dev_id_set.intersection(test_id_set)

print("=" * 60)
print("KIỂM TRA ID GIỮA HAI TẬP")
print("=" * 60)

print(f"Development ID: {len(dev_id_set):,}")
print(f"Independent Test ID: {len(test_id_set):,}")
print(f"Số ID trùng giữa hai tập: {len(overlap_ids)}")
print(f"Tổng ID sau chia: {len(dev_id_set | test_id_set):,}")

assert len(overlap_ids) == 0
assert len(dev_id_set | test_id_set) == 10000
assert len(dev_id_set) == 9000
assert len(test_id_set) == 1000

print("\nPASS: Không có ID trùng giữa hai tập.")

In [ ]:
# ==========================================
# KIỂM TRA PHÂN BỐ 9 NHÃN
# ==========================================

label_distribution = lib.pd.DataFrame({
    "Nhãn": choice_cols,
    "Full_Count": df[choice_cols].sum().values,
    "Full_Rate_%": df[choice_cols].mean().values * 100,
    "Dev_Count": df_dev[choice_cols].sum().values,
    "Dev_Rate_%": df_dev[choice_cols].mean().values * 100,
    "Test_Count": df_test[choice_cols].sum().values,
    "Test_Rate_%": df_test[choice_cols].mean().values * 100
})

label_distribution["Chênh_lệch_%"] = (
    label_distribution["Dev_Rate_%"]
    - label_distribution["Test_Rate_%"]
).abs()

numeric_cols = label_distribution.select_dtypes(
    include="number"
).columns

label_distribution[numeric_cols] = (
    label_distribution[numeric_cols].round(2)
)

print("=" * 60)
print("PHÂN BỐ 9 NHÃN GIỮA CÁC TẬP")
print("=" * 60)

display(label_distribution)

In [ ]:
# ==========================================
# KIỂM TRA PHÂN BỐ 18 TỔ HỢP MÔN
# ==========================================

def get_combinations(data):
    return data[choice_cols].astype(int).apply(
        lambda row: tuple(
            col for col, value in row.items() if value == 1
        ),
        axis=1
    )

full_combos = get_combinations(df)
dev_combos = get_combinations(df_dev)
test_combos = get_combinations(df_test)

# Lấy đầy đủ các tổ hợp từ dữ liệu gốc
all_combos = list(full_combos.value_counts().index)

combo_report = lib.pd.DataFrame({
    "Tổ hợp môn": [" + ".join(combo) for combo in all_combos],
    "Full_Count": full_combos.value_counts().reindex(
        all_combos, fill_value=0
    ).values,
    "Dev_Count": dev_combos.value_counts().reindex(
        all_combos, fill_value=0
    ).values,
    "Test_Count": test_combos.value_counts().reindex(
        all_combos, fill_value=0
    ).values
})

combo_report["Dev_%"] = (
    combo_report["Dev_Count"] / len(df_dev) * 100
).round(2)

combo_report["Test_%"] = (
    combo_report["Test_Count"] / len(df_test) * 100
).round(2)

print("=" * 60)
print("PHÂN BỐ TỔ HỢP MÔN")
print("=" * 60)

print(f"Số tổ hợp trong toàn bộ dữ liệu: {full_combos.nunique()}")
print(f"Số tổ hợp trong Development: {dev_combos.nunique()}")
print(f"Số tổ hợp trong Independent Test: {test_combos.nunique()}")

assert full_combos.nunique() == 18

display(combo_report)

In [ ]:
# ==========================================
# LƯU DEVELOPMENT / INDEPENDENT TEST
# ==========================================

split_dir = lib.Path("../data/data-splits")
split_dir.mkdir(parents=True, exist_ok=True)

# Các tập đặc trưng và nhãn
X_dev = df_dev[score_cols].copy()
y_dev = df_dev[choice_cols].copy()

X_test = df_test[score_cols].copy()
y_test = df_test[choice_cols].copy()

# Danh sách ID tương ứng
ids_dev = df_dev[[id_col]].copy()
ids_test = df_test[[id_col]].copy()

# Lưu dữ liệu
datasets = {
    "development.csv": df_dev,
    "independent_test.csv": df_test,
    "X_development.csv": X_dev,
    "y_development.csv": y_dev,
    "X_independent_test.csv": X_test,
    "y_independent_test.csv": y_test,
    "development_ids.csv": ids_dev,
}

for filename, data in datasets.items():

    output_path = split_dir / filename

    if output_path.exists():
        existing = lib.pd.read_csv(
            output_path, encoding="utf-8-sig"
        )
        lib.pd.testing.assert_frame_equal(
            existing, data.reset_index(drop=True),
            check_dtype=False,
            check_exact=False,
            atol=1e-10,
            rtol=0
        )
        print(f"Đã xác nhận, giữ nguyên: {filename}")
    else:
        data.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )
        print(f"Đã lưu: {filename}")

# Kiểm tra kích thước
assert X_dev.shape == (9000, 96)
assert y_dev.shape == (9000, 9)
assert X_test.shape == (1000, 96)
assert y_test.shape == (1000, 9)

print("\nPASS: Dữ liệu đã được lưu và kiểm tra.")
print(f"Thư mục: {split_dir.resolve()}")

In [ ]:
# ==========================================
# ĐÓNG BĂNG INDEPENDENT TEST
# CHỈ SỬ DỤNG DEVELOPMENT CHO THỰC NGHIỆM
# ==========================================

# Tải tập Development làm nguồn cho các thí nghiệm
dev_data = lib.pd.read_csv(
    split_dir / "development.csv",
    encoding="utf-8-sig"
)

# Tập dữ liệu dùng cho Cross-Validation và tối ưu
X = dev_data[score_cols].copy()
y = dev_data[choice_cols].copy()
student_ids = dev_data[id_col].copy()

# Kiểm tra không chứa ID Independent Test
frozen_test_ids = set(
    lib.pd.read_csv(
        split_dir / "independent_test_ids.csv",
        encoding="utf-8-sig"
    )[id_col]
)

assert not set(student_ids).intersection(frozen_test_ids)
assert X.shape == (9000, 96)
assert y.shape == (9000, 9)

print("=" * 60)
print("DEVELOPMENT DATA - MODEL EXPERIMENTS")
print("=" * 60)

print(f"Random Seed: {RANDOM_SEED}")
print(f"X: {X.shape}")
print(f"y: {y.shape}")
print(f"Số ID Independent Test trong Development: 0")

print("\nPASS: Chỉ sử dụng 9.000 mẫu Development cho thực nghiệm.")
print("Independent Test được giữ riêng cho đánh giá cuối cùng.")